# 02 — CH4 Association & Hotspot Labeling
Vellakkal Dumpsite methane hotspot project.

Associates each genuinely exported Sentinel-2 patch (from Batch 1) with a QA-passing Sentinel-5P CH4 retrieval, computes a **local background-subtracted enhancement**, and assigns the primary hotspot label from that enhancement. The legacy global 75th-percentile label is also computed, but only as a documented reference — not as the training target.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/methane-hotspot-detection'
import os
os.makedirs(f'{PROJECT_ROOT}/data/processed', exist_ok=True)

In [ ]:
!pip install -q earthengine-api pandas

In [ ]:
import ee
GEE_PROJECT = 'YOUR-GEE-PROJECT-ID'  # <-- set this, same project as Batch 1
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

## Set the Vellakkal dumpsite reference coordinates
Used as the center point for the local CH4 background estimate (25 km radius, wider time window). Update with the actual dumpsite centroid if different from the dataset's approximate center.

In [ ]:
import pandas as pd
raw = pd.read_csv(f'{PROJECT_ROOT}/data/raw/original_gee_export.csv')
SITE_LAT = raw['latitude'].mean()
SITE_LON = raw['longitude'].mean()
print(f'Using site center: lat={SITE_LAT:.6f}, lon={SITE_LON:.6f}')
print('Replace with the precise dumpsite centroid if you have it, for a more accurate background ring.')

## Run CH4 association + labeling
Requires `extraction_log.csv` from Batch 1 with `status == 'COMPLETED'` rows (real exported patches only).

In [ ]:
import sys
sys.path.append(f'{PROJECT_ROOT}/src')

!python {PROJECT_ROOT}/src/ch4_labeling.py \
    --extraction-log {PROJECT_ROOT}/data/raw/extraction_log.csv \
    --gee-project {GEE_PROJECT} \
    --out {PROJECT_ROOT}/data/processed/metadata.csv \
    --site-lon {SITE_LON} \
    --site-lat {SITE_LAT}

## Review label distribution and compare methods
Check how the enhancement-based label differs from the legacy percentile label, and confirm the number of patches that had to be dropped for missing/low-QA CH4.

In [ ]:
meta = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata.csv')
print('Total patches:', len(meta))
print('Missing CH4 (to be dropped before training):', meta['ch4_value'].isna().sum())
print()
print('Primary (enhancement-based) label distribution:')
print(meta['label_enhancement'].value_counts(dropna=False, normalize=True))
print()
print('Reference-only 75th-percentile label distribution:')
print(meta['label_percentile75'].value_counts(dropna=False, normalize=True))
print()
print('Agreement between the two labeling methods:')
valid = meta.dropna(subset=['label_enhancement', 'label_percentile75'])
agreement = (valid['label_enhancement'] == valid['label_percentile75']).mean()
print(f'{agreement:.1%} of patches agree between enhancement-based and percentile-based labels')

## Check CH4 pixel sharing (leakage awareness)
Because TROPOMI's footprint is far coarser than a Sentinel-2 patch, many patches will share the same `ch4_pixel_id`. This is expected and must be carried into Batch 4's spatial-block splitting so that patches sharing a CH4 source don't end up split across train/val/test.

In [ ]:
shared = meta.groupby('ch4_pixel_id')['patch_id'].count().sort_values(ascending=False)
print('Patches per shared CH4 pixel (top 10):')
print(shared.head(10))
print(f'\nUnique CH4 source pixels used: {meta["ch4_pixel_id"].nunique()}')